# RQ2 · Theil Decomposition of Electricity Sufficiency Inequality

This notebook reproduces the RQ2 analysis from the cleaned 2015 sufficiency dataset. It is written as a publishable analysis notebook: every core table and figure used for RQ2 is generated below from the input data.

**Research question**

> How much of global electricity sufficiency inequality is attributable to spatial levels, and which within-country social attributes explain the remaining subnational inequality?

**Primary outcome**: `sufficiency_ratio = EC2015_pc / 300`

**Weight**: `population`

**Main method**: population-weighted Theil T decomposition.


## 1. Setup

The notebook reads the cleaned RQ1 sufficiency dataset and writes RQ2 outputs into a dedicated results folder. The visualization section intentionally keeps only the three final RQ2 figures to avoid version clutter.

In [ ]:
from pathlib import Path
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import to_rgb, to_hex

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_DIR = Path("/Users/nico/Research/Energy sufficiency")
DATA_PATH = BASE_DIR / "Data" / "global_SE_EC_sufficiency_2015.csv"
PROCESS_DIR = BASE_DIR / "Process"
RESULTS_DIR = BASE_DIR / "Results" / "RQ2_theil_decomposition"
TABLE_DIR = RESULTS_DIR / "Tables"
FIG_DIR = RESULTS_DIR / "Figures"

for directory in [PROCESS_DIR, RESULTS_DIR, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FLOOR_THRESHOLD = 300
PRIMARY_OUTCOME = "sufficiency_ratio"
ROBUSTNESS_OUTCOME = "sufficiency_gap"
WEIGHT_COL = "population"

# Only final RQ2 figures should remain in the main figure folder.
for png in FIG_DIR.glob("*.png"):
    png.unlink()
for old_option_dir in [FIG_DIR / "Social_decomposition_options", FIG_DIR / "Cross_group_options"]:
    if old_option_dir.exists():
        shutil.rmtree(old_option_dir)


## 2. Region and Factor Definitions

The seven analytical regions and colors follow the RQ1 mapping/descriptive analysis. Social decomposition uses country-internal terciles. Cross-groups are theory-guided and focus on rurality, SES, education, gender composition, age, and household size.

In [ ]:
REGION_ORDER = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania",
]

SHORT_REGION_LABELS = {
    "Sub-Saharan Africa": "Sub-Saharan\nAfrica",
    "Northern Africa and Western Asia": "N. Africa\n& W. Asia",
    "South Asia": "South\nAsia",
    "Eastern and South-eastern Asia": "E. & SE\nAsia",
    "Europe": "Europe",
    "Latin America and the Caribbean": "Latin America\n& Caribbean",
    "Northern America and Oceania": "N. America\n& Oceania",
}

REGION_COLORS = {
    "Sub-Saharan Africa": "#53a8e1",
    "Northern Africa and Western Asia": "#ffa74f",
    "South Asia": "#ed5f5f",
    "Eastern and South-eastern Asia": "#9a72c7",
    "Europe": "#afd3e7",
    "Latin America and the Caribbean": "#fca3a3",
    "Northern America and Oceania": "#fdcd8c",
}

SOUTH_ASIA = {"AFG", "BGD", "BTN", "IND", "IRN", "LKA", "MDV", "NPL", "PAK"}
EAST_SE_ASIA = {"BRN", "CHN", "IDN", "JPN", "KHM", "KOR", "LAO", "MMR", "MNG", "MYS", "PHL", "PRK", "SGP", "THA", "TLS", "VNM"}
NORTH_AFRICA_WEST_ASIA = {"DZA", "BHR", "EGY", "IRQ", "ISR", "JOR", "KWT", "LBN", "LBY", "MAR", "OMN", "PSE", "QAT", "SAU", "SYR", "TUN", "TUR", "ARE", "YEM", "ARM", "AZE", "CYP", "GEO", "KAZ", "KGZ", "TJK", "TKM", "UZB"}
LATIN_AMERICA_CARIBBEAN = {"ATG", "ARG", "BHS", "BRB", "BLZ", "BOL", "BRA", "CHL", "COL", "CRI", "CUB", "DMA", "DOM", "ECU", "SLV", "GRD", "GTM", "GUY", "HTI", "HND", "JAM", "MEX", "NIC", "PAN", "PRY", "PER", "KNA", "LCA", "VCT", "SUR", "TTO", "URY", "VEN"}
NORTH_AMERICA_OCEANIA = {"AUS", "CAN", "FSM", "FJI", "KIR", "MHL", "NRU", "NZL", "PLW", "PNG", "SLB", "TON", "TUV", "USA", "VUT", "WSM"}
EUROPE = {"ALB", "AND", "AUT", "BEL", "BGR", "BIH", "BLR", "CHE", "CZE", "DEU", "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HRV", "HUN", "IRL", "ISL", "ITA", "LTU", "LUX", "LVA", "MDA", "MKD", "MLT", "MNE", "NLD", "NOR", "POL", "PRT", "ROU", "RUS", "SRB", "SVK", "SVN", "SWE", "UKR"}

def assign_analytical_region(iso_code):
    if pd.isna(iso_code):
        return np.nan
    iso = str(iso_code).upper()[:3]
    if iso in SOUTH_ASIA:
        return "South Asia"
    if iso in EAST_SE_ASIA:
        return "Eastern and South-eastern Asia"
    if iso in NORTH_AFRICA_WEST_ASIA:
        return "Northern Africa and Western Asia"
    if iso in LATIN_AMERICA_CARIBBEAN:
        return "Latin America and the Caribbean"
    if iso in NORTH_AMERICA_OCEANIA:
        return "Northern America and Oceania"
    if iso in EUROPE:
        return "Europe"
    return "Sub-Saharan Africa"

SINGLE_FACTORS = {
    "SES": {"source_col": "SES_raw", "group_col": "ses_group", "type": "single"},
    "Rurality": {"source_col": "rural", "group_col": "rural_group", "type": "single"},
    "Education": {"source_col": "education", "group_col": "education_group", "type": "single"},
    "Female share": {"source_col": "female_share", "group_col": "female_group", "type": "single"},
    "Age": {"source_col": "age", "group_col": "age_group", "type": "single"},
    "Household size": {"source_col": "Hhsize", "group_col": "hhsize_group", "type": "single"},
}

CROSS_FACTORS = {
    "Rurality x SES": {"parts": ["rural_group", "ses_group"], "group_col": "rural_x_ses_group", "type": "cross"},
    "Rurality x Education": {"parts": ["rural_group", "education_group"], "group_col": "rural_x_education_group", "type": "cross"},
    "Female share x SES": {"parts": ["female_group", "ses_group"], "group_col": "female_x_ses_group", "type": "cross"},
    "Female share x Education": {"parts": ["female_group", "education_group"], "group_col": "female_x_education_group", "type": "cross"},
    "SES x Education": {"parts": ["ses_group", "education_group"], "group_col": "ses_x_education_group", "type": "cross"},
    "Age x Female share": {"parts": ["age_group", "female_group"], "group_col": "age_x_female_group", "type": "cross"},
    "SES x Household size": {"parts": ["ses_group", "hhsize_group"], "group_col": "ses_x_hhsize_group", "type": "cross"},
    "Female share x Rurality": {"parts": ["female_group", "rural_group"], "group_col": "female_x_rural_group", "type": "cross"},
}

FACTOR_META = {**SINGLE_FACTORS, **CROSS_FACTORS}
SINGLE_ORDER = list(SINGLE_FACTORS.keys())
CROSS_ORDER = list(CROSS_FACTORS.keys())
FACTOR_ORDER = SINGLE_ORDER + CROSS_ORDER

FACTOR_LABELS = {
    "SES": "SES\n(income/wealth)",
    "Rurality": "Rurality",
    "Education": "Education",
    "Female share": "Female\nshare",
    "Age": "Age",
    "Household size": "Household\nsize",
    "Rurality x SES": "Rurality\nx SES",
    "Rurality x Education": "Rurality\nx Education",
    "Female share x SES": "Female share\nx SES",
    "Female share x Education": "Female share\nx Education",
    "SES x Education": "SES\nx Education",
    "Age x Female share": "Age\nx Female share",
    "SES x Household size": "SES\nx Household size",
    "Female share x Rurality": "Female share\nx Rurality",
}


## 3. Helper Functions

The Theil T formulas below use population weights. Zero-valued outcomes are allowed for `sufficiency_gap` robustness; the Theil term follows the standard limit `0 × log(0) = 0`. The primary outcome `sufficiency_ratio` is strictly positive after cleaning.

In [ ]:
def clean_numeric(series):
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)

def weighted_mean(values, weights):
    x = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not mask.any():
        return np.nan
    return float(np.sum(x[mask] * w[mask]) / np.sum(w[mask]))

def weighted_quantile(values, weights, qs):
    x = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    if len(x) == 0:
        return [np.nan] * len(qs)
    order = np.argsort(x)
    x, w = x[order], w[order]
    cumw = np.cumsum(w) / np.sum(w)
    return [float(np.interp(q, cumw, x)) for q in qs]

def weighted_tercile(values, weights):
    vals = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    out = pd.Series(pd.NA, index=vals.index, dtype="object")
    mask = vals.notna() & w.notna() & (w > 0)
    if mask.sum() < 2 or vals[mask].nunique(dropna=True) < 2:
        return out
    q1, q2 = weighted_quantile(vals[mask], w[mask], [1 / 3, 2 / 3])
    if not np.isfinite(q1) or not np.isfinite(q2):
        return out
    if q1 >= q2:
        ranks = vals[mask].rank(method="first")
        try:
            out.loc[mask] = pd.qcut(ranks, 3, labels=["low", "middle", "high"]).astype("object")
        except ValueError:
            return out
        return out
    out.loc[mask & (vals <= q1)] = "low"
    out.loc[mask & (vals > q1) & (vals <= q2)] = "middle"
    out.loc[mask & (vals > q2)] = "high"
    return out

def theil_t(frame, value_col, weight_col=WEIGHT_COL):
    x = clean_numeric(frame[value_col])
    w = clean_numeric(frame[weight_col])
    mask = x.notna() & w.notna() & (w > 0) & (x >= 0)
    x = x[mask].to_numpy(dtype=float)
    w = w[mask].to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    total_weight = w.sum()
    mean_value = (w * x).sum() / total_weight
    if not np.isfinite(mean_value) or mean_value <= 0:
        return np.nan
    ratio = x / mean_value
    terms = np.zeros_like(ratio, dtype=float)
    positive = ratio > 0
    terms[positive] = (w[positive] / total_weight) * ratio[positive] * np.log(ratio[positive])
    return float(terms.sum())

def group_theil_decomposition(frame, group_col, value_col, weight_col=WEIGHT_COL):
    x = clean_numeric(frame[value_col])
    w = clean_numeric(frame[weight_col])
    group = frame[group_col]
    mask = x.notna() & w.notna() & (w > 0) & (x >= 0) & group.notna()
    d = pd.DataFrame({"x": x[mask], "w": w[mask], "group": group[mask]})
    if d.empty:
        return {"total": np.nan, "between": np.nan, "within": np.nan, "n_obs": 0, "n_groups": 0}
    total_weight = d["w"].sum()
    mean_value = (d["w"] * d["x"]).sum() / total_weight
    if not np.isfinite(mean_value) or mean_value <= 0:
        return {"total": np.nan, "between": np.nan, "within": np.nan, "n_obs": len(d), "n_groups": d["group"].nunique()}

    total = theil_t(d.rename(columns={"x": value_col, "w": weight_col}), value_col, weight_col)
    between = 0.0
    within = 0.0
    for _, sub in d.groupby("group", dropna=True):
        group_weight = sub["w"].sum()
        group_mean = (sub["w"] * sub["x"]).sum() / group_weight if group_weight > 0 else np.nan
        group_theil = theil_t(sub.rename(columns={"x": value_col, "w": weight_col}), value_col, weight_col)
        scale = (group_weight / total_weight) * (group_mean / mean_value) if np.isfinite(group_mean) else np.nan
        group_between = 0.0 if (not np.isfinite(group_mean) or group_mean <= 0) else scale * np.log(group_mean / mean_value)
        group_within = 0.0 if not np.isfinite(group_theil) else scale * group_theil
        between += group_between
        within += group_within
    return {"total": float(total), "between": float(between), "within": float(within), "n_obs": len(d), "n_groups": d["group"].nunique()}

def mix(color_a, color_b, amount):
    a = np.array(to_rgb(color_a))
    b = np.array(to_rgb(color_b))
    return to_hex(a * (1 - amount) + b * amount)

def lighten(color, amount=0.55):
    return mix(color, "#ffffff", amount)

def darken(color, amount=0.22):
    return mix(color, "#111111", amount)

def bubble_size(value, max_value):
    return 70 + (value / max_value) ** 1.45 * 1650


## 4. Load and Clean Analysis Data

This step constructs the RQ2 base table. `SES_raw` follows the previous analysis convention: use `income` when valid, otherwise use `wealth`. The `gender` column is treated as `female_share` following the current research design language.

In [ ]:
raw = pd.read_csv(DATA_PATH)

for col in [
    "population", "EC2015_sum", "EC2015_pc", "sufficiency_ratio", "sufficiency_gap",
    "sufficiency_deficit", "sufficiency_excess", "income", "wealth", "rural",
    "education", "age", "gender", "Hhsize"
]:
    if col in raw.columns:
        raw[col] = clean_numeric(raw[col])

raw["iso_code_filled"] = raw["iso_code"].astype("object")
missing_iso = raw["iso_code_filled"].isna()
raw.loc[missing_iso, "iso_code_filled"] = raw.loc[missing_iso, "GDLcode"].astype(str).str[:3].str.upper()
raw["country_filled"] = raw["country"].astype("object")
raw.loc[raw["country_filled"].isna(), "country_filled"] = raw.loc[raw["country_filled"].isna(), "iso_code_filled"]
raw["analytical_region"] = raw["iso_code_filled"].apply(assign_analytical_region)
raw["SES_raw"] = raw["income"].where(raw["income"].notna() & (raw["income"] >= 0), raw["wealth"].where(raw["wealth"].notna() & (raw["wealth"] >= 0)))
raw["female_share"] = raw["gender"]

def make_valid_base(df, value_col, require_positive=False):
    d = df.copy()
    d[value_col] = clean_numeric(d[value_col])
    d[WEIGHT_COL] = clean_numeric(d[WEIGHT_COL])
    mask = d[WEIGHT_COL].notna() & (d[WEIGHT_COL] > 0) & d[value_col].notna() & (d[value_col] >= 0)
    if require_positive:
        mask = mask & (d[value_col] > 0)
    mask = mask & d["analytical_region"].notna() & d["iso_code_filled"].notna()
    return d.loc[mask].copy()

base = make_valid_base(raw, PRIMARY_OUTCOME, require_positive=True)
base.to_csv(PROCESS_DIR / "rq2_analysis_base_sufficiency_ratio_2015.csv", index=False)

diagnostics = {
    "input_data": str(DATA_PATH),
    "n_rows_raw": int(len(raw)),
    "n_valid_primary": int(len(base)),
    "n_countries_valid": int(base["iso_code_filled"].nunique()),
    "n_regions_valid": int(base["analytical_region"].nunique()),
    "missing_iso_filled_from_gdlcode": int(missing_iso.sum()),
    "primary_outcome": PRIMARY_OUTCOME,
    "weight": WEIGHT_COL,
}
(PROCESS_DIR / "rq2_input_diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")

diagnostics


## 5. Spatial Theil Decomposition

The global decomposition separates Theil T into three components:

```text
Total Theil = between regions + between countries within regions + within countries
```

For each region, the regional Theil is decomposed into between-country and within-country components.

In [ ]:
def spatial_decomposition(data, value_col):
    total = theil_t(data, value_col)
    region_dec = group_theil_decomposition(data, "analytical_region", value_col)

    between_region = region_dec["between"]
    between_country_within_region = 0.0
    within_country = 0.0

    global_weight = data[WEIGHT_COL].sum()
    global_mean = weighted_mean(data[value_col], data[WEIGHT_COL])
    region_rows = []
    country_rows = []

    for region in REGION_ORDER:
        region_data = data[data["analytical_region"] == region].copy()
        if region_data.empty:
            continue
        region_dec = group_theil_decomposition(region_data, "iso_code_filled", value_col)
        region_weight = region_data[WEIGHT_COL].sum()
        region_mean = weighted_mean(region_data[value_col], region_data[WEIGHT_COL])
        region_scale = (region_weight / global_weight) * (region_mean / global_mean) if global_weight > 0 and global_mean > 0 else np.nan

        embedded_between_country = region_scale * region_dec["between"] if np.isfinite(region_scale) and np.isfinite(region_dec["between"]) else np.nan
        embedded_within_country = region_scale * region_dec["within"] if np.isfinite(region_scale) and np.isfinite(region_dec["within"]) else np.nan
        between_country_within_region += 0.0 if not np.isfinite(embedded_between_country) else embedded_between_country
        within_country += 0.0 if not np.isfinite(embedded_within_country) else embedded_within_country

        region_rows.append({
            "analytical_region": region,
            "n_gdl": int(len(region_data)),
            "n_countries": int(region_data["iso_code_filled"].nunique()),
            "population": float(region_weight),
            "mean_value": region_mean,
            "regional_theil_total": region_dec["total"],
            "between_country": region_dec["between"],
            "within_country": region_dec["within"],
            "between_country_pct_of_regional_theil": region_dec["between"] / region_dec["total"] if np.isfinite(region_dec["total"]) and region_dec["total"] > 0 else np.nan,
            "within_country_pct_of_regional_theil": region_dec["within"] / region_dec["total"] if np.isfinite(region_dec["total"]) and region_dec["total"] > 0 else np.nan,
            "embedded_between_country_component_global": embedded_between_country,
            "embedded_within_country_component_global": embedded_within_country,
        })

        for iso, country_data in region_data.groupby("iso_code_filled", dropna=False):
            country_theil = theil_t(country_data, value_col)
            country_weight = country_data[WEIGHT_COL].sum()
            country_mean = weighted_mean(country_data[value_col], country_data[WEIGHT_COL])
            country_scale = (country_weight / global_weight) * (country_mean / global_mean) if global_weight > 0 and global_mean > 0 else np.nan
            embedded_country = country_scale * country_theil if np.isfinite(country_scale) and np.isfinite(country_theil) else np.nan
            country_rows.append({
                "iso_code": iso,
                "country": country_data["country_filled"].iloc[0],
                "analytical_region": region,
                "n_gdl": int(len(country_data)),
                "population": float(country_weight),
                "mean_value": country_mean,
                "country_theil": country_theil,
                "embedded_within_country_component_global": embedded_country,
                "embedded_within_country_pct_global_total": embedded_country / total if np.isfinite(embedded_country) and np.isfinite(total) and total > 0 else np.nan,
            })

    global_rows = []
    for component, value in [
        ("between_region", between_region),
        ("between_country_within_region", between_country_within_region),
        ("within_country", within_country),
    ]:
        global_rows.append({
            "outcome": value_col,
            "component": component,
            "theil_component": value,
            "share_of_total_theil": value / total if np.isfinite(total) and total > 0 else np.nan,
            "total_theil": total,
        })

    return pd.DataFrame(global_rows), pd.DataFrame(region_rows), pd.DataFrame(country_rows)

global_spatial, region_spatial, country_spatial = spatial_decomposition(base, PRIMARY_OUTCOME)

global_spatial.to_csv(TABLE_DIR / "rq2_spatial_decomposition_global.csv", index=False)
region_spatial.to_csv(TABLE_DIR / "rq2_spatial_decomposition_by_region.csv", index=False)
country_spatial.to_csv(TABLE_DIR / "rq2_country_within_contribution_list.csv", index=False)

print("Global component sum check:", global_spatial["theil_component"].sum(), "vs", global_spatial["total_theil"].iloc[0])
global_spatial


## 6. Country-Internal Social Decomposition

Social decomposition is performed **within countries** to avoid interpreting country-relative variables, such as SES, as globally absolute ranks. Each social attribute is split into population-weighted country-internal terciles. Cross-groups are then built from those country-internal terciles.

In [ ]:
work = base.copy()

# Country-internal terciles for single attributes.
for iso, idx in work.groupby("iso_code_filled").groups.items():
    idx = list(idx)
    for factor, meta in SINGLE_FACTORS.items():
        work.loc[idx, meta["group_col"]] = weighted_tercile(work.loc[idx, meta["source_col"]], work.loc[idx, WEIGHT_COL]).values

# Theory-guided cross-groups.
for factor, meta in CROSS_FACTORS.items():
    col_a, col_b = meta["parts"]
    group_col = meta["group_col"]
    valid = work[col_a].notna() & work[col_b].notna()
    work[group_col] = pd.NA
    work.loc[valid, group_col] = work.loc[valid, col_a].astype(str) + " x " + work.loc[valid, col_b].astype(str)

compact_cols = [
    "GDLcode", "iso_code_filled", "country_filled", "analytical_region", WEIGHT_COL,
    PRIMARY_OUTCOME, ROBUSTNESS_OUTCOME, "EC2015_pc", "SES_raw", "rural", "education",
    "female_share", "age", "Hhsize"
] + [meta["group_col"] for meta in FACTOR_META.values()]
work[compact_cols].to_csv(PROCESS_DIR / "rq2_social_grouped_analysis_base_2015.csv", index=False)

work[compact_cols].head()


In [ ]:
global_within_component = global_spatial.loc[global_spatial["component"] == "within_country", "theil_component"].iloc[0]
global_total_theil = global_spatial["total_theil"].iloc[0]
global_weight = work[WEIGHT_COL].sum()
global_mean = weighted_mean(work[PRIMARY_OUTCOME], work[WEIGHT_COL])

country_long_rows = []
country_summary_rows = []

for (iso, country, region), country_data in work.groupby(["iso_code_filled", "country_filled", "analytical_region"], dropna=False):
    country_theil = theil_t(country_data, PRIMARY_OUTCOME)
    country_weight = country_data[WEIGHT_COL].sum()
    country_mean = weighted_mean(country_data[PRIMARY_OUTCOME], country_data[WEIGHT_COL])
    country_scale = (country_weight / global_weight) * (country_mean / global_mean) if global_weight > 0 and global_mean > 0 and np.isfinite(country_mean) else np.nan
    embedded_country = country_scale * country_theil if np.isfinite(country_scale) and np.isfinite(country_theil) else np.nan

    base_row = {
        "iso_code": iso,
        "country": country,
        "analytical_region": region,
        "n_gdl_valid": int(len(country_data)),
        "population": float(country_weight),
        "mean_sufficiency_ratio": country_mean,
        "mean_sufficiency_gap": weighted_mean(country_data[ROBUSTNESS_OUTCOME], country_data[WEIGHT_COL]),
        "mean_sufficiency_deficit": weighted_mean(country_data["sufficiency_deficit"], country_data[WEIGHT_COL]),
        "country_theil_sufficiency_ratio": country_theil,
        "embedded_within_country_component_global": embedded_country,
        "embedded_within_country_pct_global_total": embedded_country / global_total_theil if np.isfinite(embedded_country) and global_total_theil > 0 else np.nan,
    }

    country_factor_values = {}
    for factor in FACTOR_ORDER:
        meta = FACTOR_META[factor]
        dec = group_theil_decomposition(country_data, meta["group_col"], PRIMARY_OUTCOME)
        eligible = bool(np.isfinite(dec["total"]) and dec["n_groups"] >= 2 and np.isfinite(country_theil) and country_theil > 0)
        between_share = dec["between"] / country_theil if eligible else np.nan
        embedded_between_global = country_scale * dec["between"] if eligible and np.isfinite(country_scale) else np.nan

        country_long_rows.append({
            **base_row,
            "factor": factor,
            "factor_label": FACTOR_LABELS[factor].replace("\n", " "),
            "factor_type": meta["type"],
            "n_groups_observed": int(dec["n_groups"]),
            "theil_between_social_groups": dec["between"] if eligible else np.nan,
            "theil_within_social_groups": dec["within"] if eligible else np.nan,
            "between_share_of_country_theil": between_share,
            "embedded_between_social_component_global": embedded_between_global,
        })
        country_factor_values[f"{factor} between_share"] = between_share

    singles = {k: v for k, v in country_factor_values.items() if k.replace(" between_share", "") in SINGLE_ORDER and pd.notna(v)}
    crosses = {k: v for k, v in country_factor_values.items() if k.replace(" between_share", "") in CROSS_ORDER and pd.notna(v)}

    def top_factor(values):
        if not values:
            return np.nan, np.nan
        key = max(values, key=values.get)
        return key.replace(" between_share", ""), values[key]

    top_single, top_single_share = top_factor(singles)
    top_cross, top_cross_share = top_factor(crosses)
    country_summary_rows.append({
        **base_row,
        "top_single_factor": top_single,
        "top_single_between_share": top_single_share,
        "top_cross_factor": top_cross,
        "top_cross_between_share": top_cross_share,
        **country_factor_values,
    })

country_social_long = pd.DataFrame(country_long_rows)
country_social_wide = pd.DataFrame(country_summary_rows)

country_social_long.to_csv(TABLE_DIR / "rq2_country_social_decomposition_long.csv", index=False)
country_social_wide.to_csv(TABLE_DIR / "rq2_country_social_decomposition_wide.csv", index=False)

country_social_long.head()


## 7. Aggregate Social Decomposition to Global and Regional Summaries

The global and regional summaries are not pooled social decompositions. They aggregate country-internal social decompositions using each country's embedded within-country Theil contribution. This preserves the interpretation of country-relative social groups.

In [ ]:
def summarize_social_decomposition(country_long, region=None):
    subset = country_long if region is None else country_long[country_long["analytical_region"] == region]
    if region is None:
        denominator_full = global_within_component
        region_label = "Global"
    else:
        row = region_spatial[region_spatial["analytical_region"] == region]
        denominator_full = row["embedded_within_country_component_global"].iloc[0] if not row.empty else np.nan
        region_label = region

    rows = []
    for factor in FACTOR_ORDER:
        f = subset[subset["factor"] == factor].copy()
        eligible = f[f["embedded_between_social_component_global"].notna() & f["embedded_within_country_component_global"].notna()]
        eligible_denominator = eligible["embedded_within_country_component_global"].sum()
        between_sum = eligible["embedded_between_social_component_global"].sum()
        rows.append({
            "analytical_region": region_label,
            "factor": factor,
            "factor_label": FACTOR_LABELS[factor].replace("\n", " "),
            "factor_type": FACTOR_META[factor]["type"],
            "n_countries_eligible": int(len(eligible)),
            "eligible_population": float(eligible.drop_duplicates(["iso_code"])["population"].sum()) if not eligible.empty else 0.0,
            "embedded_between_social_component": float(between_sum),
            "eligible_within_country_component": float(eligible_denominator),
            "full_within_country_component": float(denominator_full) if np.isfinite(denominator_full) else np.nan,
            "share_of_eligible_within_country_theil": between_sum / eligible_denominator if eligible_denominator > 0 else np.nan,
            "share_of_full_within_country_theil": between_sum / denominator_full if np.isfinite(denominator_full) and denominator_full > 0 else np.nan,
            "weighted_mean_country_between_share": np.average(eligible["between_share_of_country_theil"], weights=eligible["embedded_within_country_component_global"]) if len(eligible) > 0 and eligible["embedded_within_country_component_global"].sum() > 0 else np.nan,
        })
    return pd.DataFrame(rows)

global_social = summarize_social_decomposition(country_social_long)
region_social = pd.concat([summarize_social_decomposition(country_social_long, r) for r in REGION_ORDER], ignore_index=True)

global_social.to_csv(TABLE_DIR / "rq2_social_decomposition_global_summary.csv", index=False)
region_social.to_csv(TABLE_DIR / "rq2_social_decomposition_by_region.csv", index=False)

global_social[["factor", "factor_type", "share_of_eligible_within_country_theil", "n_countries_eligible"]]


## 8. Country Case List

This table supports result writing and example selection. It identifies countries that contribute strongly to global within-country inequality, have high country-level sufficiency inequality, have strong cross-group stratification, or have low mean sufficiency ratios.

In [ ]:
country_case = country_social_wide.copy()
country_case["within_country_contribution_rank"] = country_case["embedded_within_country_component_global"].rank(ascending=False, method="min")
country_case["country_theil_rank"] = country_case["country_theil_sufficiency_ratio"].rank(ascending=False, method="min")
country_case["top_cross_rank"] = country_case["top_cross_between_share"].rank(ascending=False, method="min")
country_case["low_ratio_rank"] = country_case["mean_sufficiency_ratio"].rank(ascending=True, method="min")
country_case["case_note"] = np.select(
    [
        country_case["within_country_contribution_rank"] <= 10,
        country_case["country_theil_rank"] <= 10,
        country_case["top_cross_rank"] <= 10,
        country_case["low_ratio_rank"] <= 10,
    ],
    [
        "Large contribution to global within-country inequality",
        "High country-level sufficiency inequality",
        "Strong cross-group social stratification",
        "Very low mean sufficiency ratio",
    ],
    default="",
)

case_cols = [
    "iso_code", "country", "analytical_region", "n_gdl_valid", "population",
    "mean_sufficiency_ratio", "mean_sufficiency_gap", "mean_sufficiency_deficit",
    "country_theil_sufficiency_ratio", "embedded_within_country_component_global",
    "embedded_within_country_pct_global_total", "top_single_factor", "top_single_between_share",
    "top_cross_factor", "top_cross_between_share", "within_country_contribution_rank",
    "country_theil_rank", "top_cross_rank", "low_ratio_rank", "case_note"
]

country_case = country_case[case_cols].sort_values(["within_country_contribution_rank", "country_theil_rank"])
country_case.to_csv(TABLE_DIR / "rq2_country_case_list.csv", index=False)

country_case.head(15)


## 9. Robustness: Sufficiency Gap Spatial Decomposition

As a supplementary outcome check, spatial decomposition is repeated for `sufficiency_gap`. The main RQ2 analysis remains based on `sufficiency_ratio`, the primary outcome in the research design.

In [ ]:
gap_base = make_valid_base(raw, ROBUSTNESS_OUTCOME, require_positive=False)
gap_global_spatial, gap_region_spatial, _ = spatial_decomposition(gap_base, ROBUSTNESS_OUTCOME)

gap_global_spatial.to_csv(TABLE_DIR / "rq2_robustness_sufficiency_gap_spatial_global.csv", index=False)
gap_region_spatial.to_csv(TABLE_DIR / "rq2_robustness_sufficiency_gap_spatial_by_region.csv", index=False)

gap_global_spatial


## 10. Final Visualizations

Only three final RQ2 figures are generated:

1. Social decomposition by region as a region-colored bubble matrix.
2. Spatial Theil decomposition combining global and regional rows.
3. Global social decomposition ordered by single attributes first and cross-groups second.

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.edgecolor": "#cfcfcf",
    "axes.linewidth": 0.9,
    "axes.titlesize": 15,
    "axes.labelsize": 11.5,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

# Figure 1: social decomposition by region, bubble matrix.
heat = region_social.pivot(index="analytical_region", columns="factor", values="share_of_eligible_within_country_theil").reindex(REGION_ORDER)[FACTOR_ORDER]
max_share = np.nanmax(heat.to_numpy(dtype=float))

fig, ax = plt.subplots(figsize=(13.2, 6.2), constrained_layout=True)
for i, region in enumerate(REGION_ORDER):
    color = REGION_COLORS[region]
    for j, factor in enumerate(FACTOR_ORDER):
        value = heat.loc[region, factor]
        if np.isfinite(value):
            ax.scatter(j, i, s=bubble_size(value, max_share), color=color, edgecolor="white", linewidth=1.0, alpha=0.94, zorder=3)
            ax.text(j, i, f"{value:.0%}", ha="center", va="center", fontsize=7.6, color="#111111", zorder=4)

ax.set_xlim(-0.65, len(FACTOR_ORDER) - 0.35)
ax.set_ylim(len(REGION_ORDER) - 0.55, -1.0)
ax.set_xticks(np.arange(len(FACTOR_ORDER)))
ax.set_xticklabels([FACTOR_LABELS[f] for f in FACTOR_ORDER], rotation=34, ha="right")
ax.set_yticks(np.arange(len(REGION_ORDER)))
ax.set_yticklabels([SHORT_REGION_LABELS[r].replace("\n", " ") for r in REGION_ORDER])
ax.grid(True, color="#eeeeee", linewidth=0.8, zorder=0)
ax.axvline(len(SINGLE_ORDER) - 0.5, color="#cfcfcf", linewidth=1.25)
ax.text(1.2, -0.82, "single attributes", ha="center", va="center", fontsize=9.6, color="#555555")
ax.text(10.8, -0.82, "cross-groups", ha="center", va="center", fontsize=9.6, color="#555555")
ax.set_title("Social decomposition by region", pad=28)
legend_levels = [0.20, 0.35, 0.50, 0.65, 0.80]
legend_handles = [ax.scatter([], [], s=bubble_size(v, max_share), color="#bdbdbd", edgecolor="white", linewidth=1.0) for v in legend_levels]
ax.legend(legend_handles, [f"{v:.0%}" for v in legend_levels], title="Share explained", frameon=False, loc="upper left", bbox_to_anchor=(1.015, 1.0), borderaxespad=0.0, labelspacing=1.25)
fig.savefig(FIG_DIR / "Social_decomposition_by_region_bubble_matrix.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# Figure 2: spatial decomposition; global row in grayscale, regions in RQ1 palette.
fig, ax = plt.subplots(figsize=(9.2, 5.9), constrained_layout=True)
y = np.arange(1 + len(REGION_ORDER))
left = 0.0
for component, label, color, text_color in [
    ("between_region", "Global: between regions", "#f2f2f2", "#222222"),
    ("between_country_within_region", "Global: between countries within regions", "#bdbdbd", "#222222"),
    ("within_country", "Global: within countries", "#4a4a4a", "white"),
]:
    value = float(global_spatial.loc[global_spatial["component"] == component, "share_of_total_theil"].iloc[0])
    ax.barh(y[0], value, left=left, color=color, edgecolor="white", linewidth=1.0, label=label)
    if value > 0.07:
        ax.text(left + value / 2, y[0], f"{value:.0%}", ha="center", va="center", color=text_color, fontsize=9.5, fontweight="bold")
    left += value

region_plot = region_spatial.set_index("analytical_region").reindex(REGION_ORDER).reset_index()
for i, region in enumerate(REGION_ORDER, start=1):
    base_color = REGION_COLORS[region]
    between_share = float(region_plot.loc[region_plot["analytical_region"] == region, "between_country_pct_of_regional_theil"].iloc[0])
    within_share = float(region_plot.loc[region_plot["analytical_region"] == region, "within_country_pct_of_regional_theil"].iloc[0])
    ax.barh(y[i], between_share, color=lighten(base_color, 0.40), edgecolor="white", linewidth=1.0)
    ax.barh(y[i], within_share, left=between_share, color=darken(base_color, 0.15), edgecolor="white", linewidth=1.0)
    if between_share > 0.11:
        ax.text(between_share / 2, y[i], f"{between_share:.0%}", ha="center", va="center", color="#222222", fontsize=8.8)
    if within_share > 0.11:
        ax.text(between_share + within_share / 2, y[i], f"{within_share:.0%}", ha="center", va="center", color="white", fontsize=8.8, fontweight="bold")

ax.set_yticks(y)
ax.set_yticklabels(["Global"] + [SHORT_REGION_LABELS[r] for r in REGION_ORDER])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("Share of Theil T")
ax.set_title("Spatial decomposition of sufficiency inequality")
ax.grid(axis="x", color="#e5e5e5", linewidth=0.8)
ax.tick_params(axis="both", direction="out", length=4, color="#333333")
legend_handles = [
    Patch(facecolor="#f2f2f2", edgecolor="#cccccc", label="Global: between regions"),
    Patch(facecolor="#bdbdbd", label="Global: between countries within regions"),
    Patch(facecolor="#4a4a4a", label="Global: within countries"),
    Patch(facecolor="#d9d9d9", edgecolor="#888888", label="Region rows: light = between countries, dark = within countries"),
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False, fontsize=9.2)
fig.savefig(FIG_DIR / "Spatial_theil_decomposition_global_and_regions.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# Figure 3: global social decomposition, single attributes first and cross-groups second.
ordered = global_social.set_index("factor").reindex(FACTOR_ORDER).reset_index()
fig, ax = plt.subplots(figsize=(7.2, 8.2), constrained_layout=True)
y = np.arange(len(ordered))
bar_colors = ["#53a8e1" if t == "single" else "#9a72c7" for t in ordered["factor_type"]]
ax.barh(y, ordered["share_of_eligible_within_country_theil"], color=bar_colors, edgecolor="white", linewidth=0.9)
ax.set_yticks(y)
ax.set_yticklabels([FACTOR_LABELS[f].replace("\n", " ") for f in ordered["factor"]])
ax.invert_yaxis()
ax.axhline(len(SINGLE_ORDER) - 0.5, color="#d0d0d0", linewidth=1.0)
max_value = ordered["share_of_eligible_within_country_theil"].max()
for i, value in enumerate(ordered["share_of_eligible_within_country_theil"]):
    ax.text(value + max_value * 0.012, i, f"{value:.1%}", va="center", fontsize=8.8)
ax.text(max_value * 0.98, (len(SINGLE_ORDER) - 1) / 2, "single attributes", ha="right", va="center", color="#4a4a4a", fontsize=9.5)
ax.text(max_value * 0.98, len(SINGLE_ORDER) + (len(CROSS_ORDER) - 1) / 2, "cross-groups", ha="right", va="center", color="#4a4a4a", fontsize=9.5)
ax.set_xlim(0, max_value * 1.18)
ax.set_xlabel("Share of eligible within-country Theil explained")
ax.set_title("Social decomposition of within-country inequality")
ax.grid(axis="x", color="#e5e5e5", linewidth=0.8)
ax.legend(handles=[Patch(facecolor="#53a8e1", label="Single attribute"), Patch(facecolor="#9a72c7", label="Cross-group")], frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0)
fig.savefig(FIG_DIR / "Social_decomposition_global_ordered.png", dpi=300, bbox_inches="tight")
plt.close(fig)

sorted(p.name for p in FIG_DIR.glob("*.png"))


## 11. Output Manifest and Checks

The manifest records the core files generated by this notebook. The numeric checks verify that the spatial decomposition components add up to the total Theil value within floating-point tolerance.

In [ ]:
manifest = {
    "research_question": "RQ2",
    "input_data": str(DATA_PATH),
    "primary_outcome": PRIMARY_OUTCOME,
    "robustness_outcome": ROBUSTNESS_OUTCOME,
    "weight": WEIGHT_COL,
    "core_tables": sorted(p.name for p in TABLE_DIR.glob("rq2_*.csv")),
    "final_figures": sorted(p.name for p in FIG_DIR.glob("*.png")),
    "single_factors": SINGLE_ORDER,
    "cross_factors": CROSS_ORDER,
    "notes": [
        "Social decomposition is performed within countries and then aggregated upward.",
        "The main figure folder intentionally contains only the three final RQ2 figures.",
    ],
}
(PROCESS_DIR / "rq2_method_metadata.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(PROCESS_DIR / "rq2_visual_final_manifest.json").write_text(json.dumps({"figures": manifest["final_figures"]}, indent=2), encoding="utf-8")

component_sum = global_spatial["theil_component"].sum()
total_theil = global_spatial["total_theil"].iloc[0]
region_sum_error = (region_spatial["between_country"] + region_spatial["within_country"] - region_spatial["regional_theil_total"]).abs().max()

checks = {
    "global_component_sum": float(component_sum),
    "global_total_theil": float(total_theil),
    "global_sum_abs_error": float(abs(component_sum - total_theil)),
    "max_regional_sum_abs_error": float(region_sum_error),
}
checks


## 12. Display Final Figures

In [ ]:
from IPython.display import Image, display

for figure_name in [
    "Social_decomposition_by_region_bubble_matrix.png",
    "Spatial_theil_decomposition_global_and_regions.png",
    "Social_decomposition_global_ordered.png",
]:
    display(Image(filename=str(FIG_DIR / figure_name)))
